In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension from 1940-01-01 to 2030-12-31 (one row per day)'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)              AS date_key,
  d                                                     AS date,
  -- Day
  day(d)                                                AS day_of_month,
  extract(DOW_ISO FROM d)                               AS day_of_week,        -- 1 = Monday ... 7 = Sunday (ISO)
  date_format(d, 'EEEE')                                AS day_name,
  date_format(d, 'EEE')                                 AS day_name_short,
  dayofyear(d)                                          AS day_of_year,
  CASE WHEN extract(DOW_ISO FROM d) IN (6, 7) THEN true ELSE false END AS is_weekend,
  CASE WHEN extract(DOW_ISO FROM d) IN (6, 7) THEN false ELSE true END AS is_weekday,
  -- Week (ISO 8601, weeks start on Monday)
  weekofyear(d)                                         AS week_of_year,
  extract(YEAROFWEEK FROM d)                            AS iso_year,
  CAST(date_trunc('WEEK', d) AS DATE)                   AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)      AS week_end_date,
  CAST(ceil(day(d) / 7.0) AS INT)                       AS week_of_month,
  -- Month
  month(d)                                              AS month,
  date_format(d, 'MMMM')                                AS month_name,
  date_format(d, 'MMM')                                 AS month_name_short,
  date_format(d, 'yyyy-MM')                             AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                 AS year_month_key,
  CAST(date_trunc('MONTH', d) AS DATE)                  AS month_start_date,
  last_day(d)                                           AS month_end_date,
  day(last_day(d))                                      AS days_in_month,
  d = CAST(date_trunc('MONTH', d) AS DATE)              AS is_month_start,
  d = last_day(d)                                       AS is_month_end,
  -- Quarter
  quarter(d)                                            AS quarter,
  concat('Q', quarter(d))                               AS quarter_name,
  concat(year(d), '-Q', quarter(d))                     AS year_quarter,
  CAST(date_trunc('QUARTER', d) AS DATE)                AS quarter_start_date,
  date_add(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), -1) AS quarter_end_date,
  datediff(d, CAST(date_trunc('QUARTER', d) AS DATE)) + 1 AS day_of_quarter,
  -- Half year
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END             AS half_year,
  concat('H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END) AS half_year_name,
  -- Year
  year(d)                                               AS year,
  CAST(date_trunc('YEAR', d) AS DATE)                   AS year_start_date,
  make_date(year(d), 12, 31)                            AS year_end_date,
  (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 AS is_leap_year,
  CAST(floor(year(d) / 10) * 10 AS INT)                 AS decade
FROM dates
ORDER BY d;

In [0]:
%sql
SELECT count(*) AS rows, count(DISTINCT date_key) AS keys, min(date) AS min_d, max(date) AS max_d,
 datediff(DATE'2030-12-31', DATE'1940-01-01')+1 AS expected,
 sum(CASE WHEN is_leap_year AND month=2 AND day_of_month=29 THEN 1 ELSE 0 END) AS feb29s,
 max(struct(date, day_name, day_of_week, week_of_year, iso_year, quarter_end_date, week_start_date)) FILTER (WHERE date = DATE'2024-12-30') AS sample_20241230,
 max(struct(date, day_name, week_of_year, iso_year)) FILTER (WHERE date = DATE'2021-01-01') AS sample_20210101
FROM adb_caresync.silver.calendar

In [0]:
%sql
SELECT *
FROM adb_caresync.silver.calendar
LIMIT 5